# Qwen BPMN Worker — procédure et narration

Ce notebook Kaggle charge **Qwen une seule fois**, puis exécute les
générateurs versionnés du projet.

Modes disponibles dans la cellule de configuration :

- `procedure` : génère uniquement la procédure ;
- `narrative` : génère uniquement la description narrative ;
- `both` : génère les deux indépendamment avec le même modèle chargé.

Les deux chaînes restent séparées :

```text
operation_contexts.json
    ├── procedure_generation
    │      └── generated_procedure.json
    │
    └── narrative_generation
           + narrative_plan.json
           └── generated_narrative.json
```

Activez **GPU** et **Internet** dans les paramètres Kaggle.


In [1]:
!pip install -q \
    "transformers>=4.51.0,<5" \
    "accelerate>=1.0" \
    "bitsandbytes>=0.45" \
    "huggingface_hub>=0.30" \
    "pydantic>=2.7,<3"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 44.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.5 MB/s eta 0:00:00


In [2]:
# Configuration

from pathlib import Path

# Choose: "procedure", "narrative", or "both".
RUN_MODE = "both"

REPO_URL = (
    "https://github.com/nour0205/"
    "bpmn-procedure-generator.git"
)
REPO_BRANCH = "main"

MODEL_NAME = "Qwen/Qwen3-8B"

PROCEDURE_PROMPT_VERSION = (
    "independent-procedure-v1.0"
)
NARRATIVE_PROMPT_VERSION = (
    "independent-narrative-v1.2"
)

# Set only when operation_contexts.json still contains
# a technical title such as Id_xxx.
PROCESS_TITLE_OVERRIDE = None
# Example:
# PROCESS_TITLE_OVERRIDE = "Suivi Commandes"

INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
REPO_DIR = (
    WORKING_DIR
    / "bpmn-procedure-generator"
)
OUTPUT_DIR = WORKING_DIR

MAX_ATTEMPTS_PER_OPERATION = 3
MAX_NEW_TOKENS_PER_OPERATION = 420

MAX_ATTEMPTS_PER_UNIT = 3
MAX_NEW_TOKENS_PER_UNIT = 650

allowed_modes = {
    "procedure",
    "narrative",
    "both",
}

if RUN_MODE not in allowed_modes:
    raise ValueError(
        "RUN_MODE must be one of: "
        "'procedure', 'narrative', 'both'."
    )

print("Run mode:", RUN_MODE)
print("Repository:", REPO_URL)
print("Branch:", REPO_BRANCH)
print("Model:", MODEL_NAME)


Run mode: both
Repository: https://github.com/nour0205/bpmn-procedure-generator.git
Branch: main
Model: Qwen/Qwen3-8B


In [3]:
# Load the version-controlled project source

import base64
import os
import shutil
import subprocess
import sys


def find_attached_project_src(
    root: Path,
):
    candidates = []

    for marker in root.rglob(
        "src/procedure_generation/"
        "__init__.py"
    ):
        src_dir = marker.parents[1]

        if (
            src_dir
            / "narrative_generation"
            / "__init__.py"
        ).exists():
            candidates.append(
                src_dir.resolve()
            )

    candidates = sorted(
        set(candidates)
    )

    if len(candidates) > 1:
        formatted = "\n".join(
            f"- {candidate}"
            for candidate in candidates
        )
        raise RuntimeError(
            "Several attached project copies "
            "were detected:\n"
            f"{formatted}\n"
            "Keep only one project dataset."
        )

    if candidates:
        return candidates[0]

    return None


project_src = find_attached_project_src(
    INPUT_ROOT
)

if project_src is None:
    github_token = None

    try:
        from kaggle_secrets import (
            UserSecretsClient,
        )

        github_token = (
            UserSecretsClient()
            .get_secret("GITHUB_TOKEN")
        )
    except Exception:
        github_token = os.environ.get(
            "GITHUB_TOKEN"
        )

    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    clone_command = [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        REPO_URL,
        str(REPO_DIR),
    ]

    if github_token:
        basic_token = base64.b64encode(
            (
                "x-access-token:"
                + github_token
            ).encode("utf-8")
        ).decode("ascii")

        clone_command = [
            "git",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: basic "
                + basic_token
            ),
            "clone",
            "--depth",
            "1",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ]

    subprocess.run(
        clone_command,
        check=True,
    )

    project_src = REPO_DIR / "src"

required_packages = [
    (
        project_src
        / "procedure_generation"
        / "__init__.py"
    ),
    (
        project_src
        / "narrative_generation"
        / "__init__.py"
    ),
]

missing_packages = [
    path
    for path in required_packages
    if not path.exists()
]

if missing_packages:
    formatted = "\n".join(
        f"- {path}"
        for path in missing_packages
    )
    raise FileNotFoundError(
        "The cloned/attached repository "
        "does not contain the required "
        "generation packages:\n"
        f"{formatted}\n"
        "Commit and push both refactors "
        "before running this notebook."
    )

sys.path.insert(
    0,
    str(project_src),
)

from procedure_generation import (
    ProcedureGenerationConfig,
    run_procedure_generation,
)
from procedure_generation.model_adapter import (
    QwenTextGenerator
    as ProcedureQwenTextGenerator,
)

from narrative_generation import (
    NarrativeGenerationConfig,
    run_narrative_generation,
)
from narrative_generation.model_adapter import (
    QwenTextGenerator
    as NarrativeQwenTextGenerator,
)

print("Project source:", project_src)


Cloning into '/kaggle/working/bpmn-procedure-generator'...


Project source: /kaggle/working/bpmn-procedure-generator/src


In [4]:
# Detect and validate input files

import json


def find_unique_input(
    filename: str,
) -> Path:
    matches = sorted(
        INPUT_ROOT.rglob(filename)
    )

    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found "
            "under /kaggle/input."
        )

    if len(matches) > 1:
        formatted = "\n".join(
            f"- {path}"
            for path in matches
        )
        raise RuntimeError(
            f"Several {filename} files "
            "were detected:\n"
            f"{formatted}\n"
            "Attach only one process input."
        )

    return matches[0]


OPERATION_CONTEXTS_PATH = (
    find_unique_input(
        "operation_contexts.json"
    )
)

input_payload = json.loads(
    OPERATION_CONTEXTS_PATH.read_text(
        encoding="utf-8"
    )
)

required_context_fields = {
    "process_id",
    "operation_count",
    "contexts",
}

if not required_context_fields.issubset(
    input_payload
):
    missing = sorted(
        required_context_fields
        - set(input_payload)
    )
    raise ValueError(
        "operation_contexts.json has an "
        "invalid schema. Missing fields: "
        + ", ".join(missing)
    )

NARRATIVE_PLAN_PATH = None

if RUN_MODE in {
    "narrative",
    "both",
}:
    NARRATIVE_PLAN_PATH = (
        find_unique_input(
            "narrative_plan.json"
        )
    )

    narrative_plan_payload = (
        json.loads(
            NARRATIVE_PLAN_PATH
            .read_text(
                encoding="utf-8"
            )
        )
    )

    plan_process_id = (
        narrative_plan_payload.get(
            "process_id"
        )
    )

    if (
        plan_process_id
        and plan_process_id
        != input_payload["process_id"]
    ):
        raise ValueError(
            "Input process IDs do not "
            "match:\n"
            "operation_contexts.json: "
            f"{input_payload['process_id']}\n"
            "narrative_plan.json: "
            f"{plan_process_id}"
        )

print(
    "Operation contexts:",
    OPERATION_CONTEXTS_PATH,
)
print(
    "Process ID:",
    input_payload["process_id"],
)
print(
    "Input title:",
    input_payload.get("title"),
)
print(
    "Operations:",
    input_payload["operation_count"],
)

if NARRATIVE_PLAN_PATH is not None:
    print(
        "Narrative plan:",
        NARRATIVE_PLAN_PATH,
    )

if (
    RUN_MODE in {
        "procedure",
        "both",
    }
    and PROCESS_TITLE_OVERRIDE is None
    and str(
        input_payload.get(
            "title",
            ""
        )
    ).startswith("Id_")
):
    print(
        "Warning: the procedure input "
        "title is technical. Set "
        "PROCESS_TITLE_OVERRIDE in the "
        "configuration cell or fix the "
        "local context exporter."
    )


Operation contexts: /kaggle/input/datasets/nourkouider05/last-test-final/operation_contexts.json
Process ID: Id_14e4a3e9-61cb-46d3-8ab0-3058189f16da
Input title: Suivi des commandes
Operations: 11
Narrative plan: /kaggle/input/datasets/nourkouider05/last-test-final/narrative_plan.json


In [5]:
# Load Qwen once in NF4 4-bit mode

import gc
import platform

import torch
import transformers
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Enable a GPU "
        "accelerator in Kaggle settings."
    )

hf_token = None

try:
    from kaggle_secrets import (
        UserSecretsClient,
    )

    hf_token = (
        UserSecretsClient()
        .get_secret("HF_TOKEN")
    )
except Exception:
    hf_token = os.environ.get(
        "HF_TOKEN"
    )

if hf_token:
    login(token=hf_token)
    print(
        "Hugging Face authentication "
        "successful."
    )
else:
    print(
        "No HF_TOKEN found. Loading "
        "continues if the model is public."
    )

os.environ[
    "HF_HUB_DOWNLOAD_TIMEOUT"
] = "600"
os.environ[
    "HF_HUB_ETAG_TIMEOUT"
] = "60"

print(
    "Python:",
    platform.python_version(),
)
print(
    "PyTorch:",
    torch.__version__,
)
print(
    "Transformers:",
    transformers.__version__,
)
print(
    "GPU:",
    torch.cuda.get_device_name(0),
)

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

quantization_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=(
            compute_dtype
        ),
        bnb_4bit_use_double_quant=True,
    )
)

tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=hf_token,
        use_fast=True,
    )
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )

model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME,
        token=hf_token,
        quantization_config=(
            quantization_config
        ),
        device_map="auto",
        torch_dtype=compute_dtype,
        low_cpu_mem_usage=True,
    )
)

model.eval()

print("Model loaded:", MODEL_NAME)
print(
    "Device map:",
    model.hf_device_map,
)

gc.collect()
torch.cuda.empty_cache()


No HF_TOKEN found. Loading continues if the model is public.
Python: 3.12.13
PyTorch: 2.10.0+cu128
Transformers: 4.57.6
GPU: Tesla T4


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-8B
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


In [6]:
# Run procedure generation when requested

procedure_result = None

if RUN_MODE in {
    "procedure",
    "both",
}:
    procedure_text_generator = (
        ProcedureQwenTextGenerator(
            model=model,
            tokenizer=tokenizer,
        )
    )

    procedure_config = (
        ProcedureGenerationConfig(
            model_name=MODEL_NAME,
            prompt_version=(
                PROCEDURE_PROMPT_VERSION
            ),
            max_attempts_per_operation=(
                MAX_ATTEMPTS_PER_OPERATION
            ),
            max_new_tokens_per_operation=(
                MAX_NEW_TOKENS_PER_OPERATION
            ),
            use_model=True,
            cache_dir=(
                OUTPUT_DIR
                / "procedure_cache"
            ),
            title_override=(
                PROCESS_TITLE_OVERRIDE
            ),
        )
    )

    procedure_result = (
        run_procedure_generation(
            operation_contexts_path=(
                OPERATION_CONTEXTS_PATH
            ),
            output_dir=OUTPUT_DIR,
            text_generator=(
                procedure_text_generator
            ),
            config=procedure_config,
        )
    )

    print(
        "Generated procedure:",
        procedure_result
        .generated_procedure_path,
    )
    print(
        "Procedure validation:",
        procedure_result
        .validation_report_path,
    )
    print(
        "Procedure preview:",
        procedure_result.preview_path,
    )

    gc.collect()
    torch.cuda.empty_cache()
else:
    print(
        "Procedure generation skipped."
    )


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[1/11] Operation 1: Lancement des commandes
[2/11] Operation 2: J+X
[3/11] Operation 3: Génération d'un reporting des commandes en cours
[4/11] Operation 4: Relancer le Fournisseur sur le portail et par téléphone
[5/11] Operation 5: Gestion des réceptions
[6/11] Operation 6: J+Y sans réponse du Fournisseur
[7/11] Operation 7: Générer les lettres de relance
[8/11] Operation 8: Valider la lettre de relance
[9/11] Operation 9: Transmettre la lettre de relance au Fournisseur via le portail et par mail
[10/11] Operation 10: Communication d'une nouvelle date de réception de la part du Fournisseur sur le portail
[11/11] Operation 11: Mettre à jour le planning de livraisons prévues
Generated procedure: /kaggle/working/generated_procedure.json
Procedure validation: /kaggle/working/procedure_validation_report.json
Procedure preview: /kaggle/working/procedure_preview.txt


In [7]:
# Run narrative generation when requested

narrative_result = None

if RUN_MODE in {
    "narrative",
    "both",
}:
    narrative_text_generator = (
        NarrativeQwenTextGenerator(
            model=model,
            tokenizer=tokenizer,
        )
    )

    narrative_config = (
        NarrativeGenerationConfig(
            model_name=MODEL_NAME,
            prompt_version=(
                NARRATIVE_PROMPT_VERSION
            ),
            max_attempts_per_unit=(
                MAX_ATTEMPTS_PER_UNIT
            ),
            max_new_tokens_per_unit=(
                MAX_NEW_TOKENS_PER_UNIT
            ),
            use_model=True,
            cache_dir=(
                OUTPUT_DIR
                / "narrative_cache"
            ),
        )
    )

    narrative_result = (
        run_narrative_generation(
            narrative_plan_path=(
                NARRATIVE_PLAN_PATH
            ),
            operation_contexts_path=(
                OPERATION_CONTEXTS_PATH
            ),
            output_dir=OUTPUT_DIR,
            text_generator=(
                narrative_text_generator
            ),
            config=narrative_config,
        )
    )

    print(
        "Generated narrative:",
        narrative_result
        .generated_narrative_path,
    )
    print(
        "Narrative validation:",
        narrative_result
        .validation_report_path,
    )
    print(
        "Narrative preview:",
        narrative_result.preview_path,
    )

    gc.collect()
    torch.cuda.empty_cache()
else:
    print(
        "Narrative generation skipped."
    )


Generated narrative: /kaggle/working/generated_narrative.json
Narrative validation: /kaggle/working/narrative_validation_report.json
Narrative preview: /kaggle/working/narrative_preview.txt


In [8]:
# Final quality summary and output manifest

run_result = {
    "mode": RUN_MODE,
    "process_id": (
        input_payload["process_id"]
    ),
    "process_title": (
        PROCESS_TITLE_OVERRIDE
        or input_payload.get("title")
    ),
    "model_name": MODEL_NAME,
    "procedure": {
        "status": "skipped",
    },
    "narrative": {
        "status": "skipped",
    },
}

if procedure_result is not None:
    procedure_report = (
        procedure_result
        .validation_report
    )
    procedure = (
        procedure_result
        .generated_procedure
    )

    procedure_summary = {
        "operation_count": (
            procedure_report[
                "operation_count"
            ]
        ),
        "missing_operation_numbers": (
            procedure_report[
                "missing_operation_numbers"
            ]
        ),
        "unknown_operation_numbers": (
            procedure_report[
                "unknown_operation_numbers"
            ]
        ),
        "fallback_operations": (
            procedure_report[
                "fallback_operations"
            ]
        ),
        "operations_requiring_validation": (
            procedure_report[
                "operations_requiring_validation"
            ]
        ),
        "placeholder_count": (
            procedure_report[
                "placeholder_count"
            ]
        ),
        "missing_note_count": (
            procedure_report[
                "missing_note_count"
            ]
        ),
        "manual_review_required": (
            procedure_report[
                "manual_review_required"
            ]
        ),
    }

    run_result["procedure"] = {
        "status": "success",
        "generated_file": (
            procedure_result
            .generated_procedure_path
            .name
        ),
        "validation_file": (
            procedure_result
            .validation_report_path
            .name
        ),
        "quality": procedure_summary,
    }

    print(
        "\n=== PROCEDURE VALIDATION ===\n"
    )
    print(
        json.dumps(
            procedure_summary,
            ensure_ascii=False,
            indent=2,
        )
    )

    for operation in procedure[
        "operations"
    ]:
        print(
            f"\n"
            f"{operation['operation_number']}. "
            f"{operation['description']}"
        )

if narrative_result is not None:
    narrative_report = (
        narrative_result
        .validation_report
    )
    narrative = (
        narrative_result
        .generated_narrative
    )

    narrative_summary = {
        "coverage": (
            narrative_report[
                "coverage_summary"
            ]
        ),
        "fallback_units": (
            narrative_report[
                "fallback_units"
            ]
        ),
        "placeholder_count": (
            narrative_report[
                "placeholder_count"
            ]
        ),
        "manual_review_required": (
            narrative_report[
                "manual_review_required"
            ]
        ),
    }

    run_result["narrative"] = {
        "status": "success",
        "generated_file": (
            narrative_result
            .generated_narrative_path
            .name
        ),
        "validation_file": (
            narrative_result
            .validation_report_path
            .name
        ),
        "quality": narrative_summary,
    }

    print(
        "\n=== NARRATIVE PREVIEW ==="
    )

    for index, paragraph in enumerate(
        narrative["paragraphs"],
        start=1,
    ):
        print(
            f"\n=== PARAGRAPHE "
            f"{index} ===\n"
        )
        print(paragraph)

    print(
        "\n=== NARRATIVE "
        "VALIDATION ===\n"
    )
    print(
        json.dumps(
            narrative_summary,
            ensure_ascii=False,
            indent=2,
        )
    )

run_result_path = (
    OUTPUT_DIR / "run_result.json"
)

run_result_path.write_text(
    json.dumps(
        run_result,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "\nRun result:",
    run_result_path,
)

print(
    "\nGenerated files:"
)

generated_names = [
    "generated_procedure.json",
    "procedure_validation_report.json",
    "procedure_preview.txt",
    "generated_narrative.json",
    "narrative_validation_report.json",
    "narrative_preview.txt",
    "run_result.json",
]

for filename in generated_names:
    path = OUTPUT_DIR / filename

    if path.exists():
        print("-", path)



=== PROCEDURE VALIDATION ===

{
  "operation_count": 11,
  "missing_operation_numbers": [],
  "unknown_operation_numbers": [],
  "fallback_operations": [],
  "operations_requiring_validation": [],
  "placeholder_count": 0,
  "missing_note_count": 0,
  "manual_review_required": false
}

1. Direction d'Approvisionnement déclenche le sous-processus « Lancement des commandes ».

2. Le processus atteint le repère temporel « J+X ».

3. Le système exécute automatiquement l’activité « Génération d'un reporting des commandes en cours ». Cette activité permet de filtrer les commandes par date, produit, etc. Elle permet également de consulter le reliquat de livraison, le statut, etc.

4. Direction d'Approvisionnement réalise l’activité « Relancer le Fournisseur sur le portail et par téléphone ».

5. Direction d'Approvisionnement déclenche le sous-processus « Gestion des réceptions ».

6. À l’événement « J+Y », aucune réponse du Fournisseur n’a été reçue.

7. Direction d'Approvisionnement réalise